In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

file_path = "default of credit card clients.xls"
df = pd.read_excel(file_path, header=1, engine="xlrd")

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

C:\Users\charl\AppData\Local\Temp\ipykernel_41100\3309474489.py:5: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  df = pd.read_excel(file_path, header=1, engine="xlrd")


Dataset shape: 30,000 rows × 25 columns


In [2]:
target_col = "default payment next month"

X = df.drop(columns=["ID", target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)
split_summary = pd.DataFrame(
    {
        "Samples": [len(y_train), len(y_test)],
        "Features": [X_train.shape[1], X_test.shape[1]],
        "Defaults": [y_train.sum(), y_test.sum()],
        "Default Rate": [f"{y_train.mean():.3%}", f"{y_test.mean():.3%}"],
    },
    index=["Train", "Test"],
)

split_summary

,Samples,Features,Defaults,Default Rate
Train,24000,23,5309,22.121%
Test,6000,23,1327,22.117%


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

baseline_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("logistic", LogisticRegression(max_iter=2000)),
    ]
)


In [4]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

param_grid = {
    "logistic__C": [0.01, 0.1, 1.0, 10.0],
}

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    estimator=baseline_model,
    param_grid=param_grid,
    scoring={"log_loss": "neg_log_loss", "roc_auc": "roc_auc"},
    cv=cv,
    refit=False,
    error_score="raise",
)

_ = search.fit(X_train, y_train)

In [6]:
cv_results = pd.DataFrame(search.cv_results_)

cv_summary = pd.DataFrame(
    {
        "C": cv_results["param_logistic__C"].astype(float),
        "Mean Log Loss": -cv_results["mean_test_log_loss"],
        "Log Loss SD": cv_results["std_test_log_loss"],
        "Mean ROC-AUC": cv_results["mean_test_roc_auc"],
        "ROC-AUC SD": cv_results["std_test_roc_auc"],
    }
).sort_values("Mean Log Loss")

best_params = cv_results.loc[cv_summary.index[0], "params"]

print(f"Best C: {best_params['logistic__C']}")
display(cv_summary.reset_index(drop=True).round(4))

Best C: 0.01


,C,Mean Log Loss,Log Loss SD,Mean ROC-AUC,ROC-AUC SD
0,0.01,0.4648,0.0052,0.7258,0.0112
1,0.10,0.4650,0.0055,0.7261,0.0111
2,1.00,0.4650,0.0055,0.7261,0.0111
3,10.00,0.4650,0.0055,0.7261,0.0111


In [8]:
from sklearn.base import clone

final_model = clone(baseline_model).set_params(**best_params)
_ = final_model.fit(X_train, y_train)

y_test_prob = final_model.predict_proba(X_test)[:, 1]

threshold = 0.50
y_test_pred = (y_test_prob >= threshold).astype(int)

In [9]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)

test_metrics = pd.Series(
    {
        "Log Loss": log_loss(y_test, y_test_prob),
        "ROC-AUC": roc_auc_score(y_test, y_test_prob),
        "Average Precision": average_precision_score(y_test, y_test_prob),
        "Accuracy": accuracy_score(y_test, y_test_pred),
        "Precision": precision_score(y_test, y_test_pred, zero_division=0),
        "Recall": recall_score(y_test, y_test_pred, zero_division=0),
        "F1": f1_score(y_test, y_test_pred, zero_division=0),
    },
    name="Test Score",
)

display(test_metrics.round(4).to_frame())

,Test Score
Log Loss,0.4713
ROC-AUC,0.7067
Average Precision,0.4926
Accuracy,0.8070
Precision,0.6882
Recall,0.2329
F1,0.3480


In [10]:
from sklearn.metrics import confusion_matrix

test_confusion_matrix = pd.DataFrame(
    confusion_matrix(y_test, y_test_pred, labels=[0, 1]),
    index=["Actual Non-Default", "Actual Default"],
    columns=["Predicted Non-Default", "Predicted Default"],
)

display(test_confusion_matrix)

,Predicted Non-Default,Predicted Default
Actual Non-Default,4533,140
Actual Default,1018,309
